# From a chrony log to stability and a noise model

This notebook takes the offsets chrony measured against one NTP server and answers three
questions:

1. how stable is the measured offset, with confidence intervals;
2. which power-law noise dominates at each averaging time;
3. what noise model (h₋₂…h₂) reproduces it, so it can drive a simulation.

The example log ships with ntpstats (`examples/data/chrony-measurements.log`); point `path` at
your own `/var/log/chrony/measurements.log` to analyse a real server.

<sub>MIT licensed, part of [ntpstats](https://github.com/thiagodefreitas/NetworkTime). Runs in CI on every change.</sub>

In [ ]:
from pathlib import Path

import numpy as np

from ntpstats import api as nt

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())

path = DATA / "chrony-measurements.log"
series = nt.load(path)
for s in series:
    print(f"{s.name}: {len(s)} samples")

`load` returns one series per server in the log. Offsets are always **reference − local**, in
seconds, whatever the daemon's own convention. `summary` gives the descriptive statistics.

In [ ]:
s = nt.load_one(path, peer="192.0.2.10")
info = nt.summary(s)
print(f"span {info['span_s'] / 3600:.1f} h, interval {info['median_interval_s']:.0f} s, gaps {info['gaps']}")
print(f"offset RMS {nt.format_seconds(info['rms'])}, 90 % range {nt.format_seconds(info['range_90'])}")
print(f"median delay {nt.format_seconds(info['delay_median'])}, outliers (5 MAD) {info['outliers_5mad']}")

## Stability with confidence intervals

`series_stability` puts irregular logs on a grid without inventing data, chooses τ₀ from the
data, and returns each deviation with a χ² interval from the exact equivalent degrees of freedom
and the noise type identified at each τ (α: 2 white PM, 1 flicker PM, 0 white FM, −1 flicker FM,
−2 random-walk FM).

In [ ]:
results = {r.kind: r for r in nt.series_stability(s, kinds=("oadev", "mdev", "tdev"), ci=0.95)}
r = results["oadev"]
print(f"tau0 = {r.tau0:g} s")
print(f"{'tau (s)':>9} {'ADEV':>10} {'95 % interval':>23} {'EDF':>7} alpha")
for tau, dev, lo, hi, edf, a in zip(r.taus, r.dev, r.lo, r.hi, r.edf, r.alpha):
    print(f"{tau:9.0f} {dev:10.3e} [{lo:10.3e}, {hi:10.3e}] {edf:7.1f} {a:+.0f}")

## Power-law noise model

`fit_noise` fits h_α to the deviations (weighted by their EDF) and bootstraps intervals. The
reduced χ² near 1 says the model explains the data within its uncertainty.

In [ ]:
fit = nt.fit_noise(s, kinds=("oadev", "mdev"), bootstrap=50, seed=1)
names = {2: "white PM", 1: "flicker PM", 0: "white FM", -1: "flicker FM", -2: "random-walk FM"}
for a in sorted(fit.h, reverse=True):
    if fit.h[a] > 0:
        print(f"h{a:+d} ({names[a]:>14}): {fit.h[a]:.3e}  [{fit.lo[a]:.3e}, {fit.hi[a]:.3e}]")
print(f"reduced chi2 = {fit.reduced_chi2:.2f}")
print("dominant noise per tau:", [names.get(a) for a in fit.dominant("oadev", r.taus)])

The model curve should run through the measured points; the shaded band is the 95 % interval.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:  # pip install "ntpstats[plot]"
    plt = None

if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.fill_between(r.taus, r.lo, r.hi, alpha=0.25, label="95 % interval")
    ax.loglog(r.taus, r.dev, "o", label="OADEV (measured)")
    grid = np.logspace(np.log10(r.taus[0]), np.log10(r.taus[-1]), 50)
    ax.loglog(grid, fit.predict("oadev", grid), "-", label="noise model")
    ax.set(xlabel="tau (s)", ylabel="Allan deviation", title=s.name)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    plt.show()

## Use the model

The fitted model becomes a simulator clock, so an algorithm can be tested on a clock with the
same noise as this server path (see the bench notebook), and with pandas installed every result
is also a dataframe.

In [ ]:
clock = nt.ClockModel(h_alpha={a: v for a, v in fit.h.items() if v > 0})
meas, truth = nt.simulate_ntp(nt.Scenario(duration=12 * 3600, poll=64, clock=clock, seed=3))
sim = nt.series_stability(truth, kinds=("oadev",))[0]
print("simulated clock ADEV at tau =", sim.taus[:4], "->", np.round(sim.dev[:4], 12))

try:
    df = s.to_pandas()
    print(df.head())
except ImportError:
    print('pip install "ntpstats[data]" for pandas')